#### Transformations are operations on RDD/DataFrame that define a new RDD/DataFrame, but are lazy (they don’t execute until an action is called).

### 1. Narrow Transformations (no shuffle, faster)

### Operate on a single partition of the parent RDD.

#### Examples:

map(func) → applies function to each element.

flatMap(func) → applies function & flattens result.

filter(func) → keeps elements that satisfy condition.

mapPartitions(func) → like map, but per partition.

mapPartitionsWithIndex(func) → adds partition index.

sample(withReplacement, fraction, seed) → samples elements.

union(otherRDD) → combines two RDDs.

intersection(otherRDD) → common elements.

distinct([numPartitions]) → removes duplicates.

pipe(command) → pipes elements to external process.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "/Users/SWAROOP/Documents/DATA ENGINEERING/Projects/Pyspark/myenv/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/Users/SWAROOP/Documents/DATA ENGINEERING/Projects/Pyspark/myenv/bin/python"

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('NarrowTransformationsInRDD').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/10 15:24:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/10 15:24:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
spark

In [5]:
# Define Spark Context object
sc = spark.sparkContext

In [6]:
sample_rdd = sc.parallelize([1,2,3,45,6.2,'hello'])

In [7]:
rdd_map = sample_rdd.map(lambda x: (x,type(x)))

In [8]:
rdd_map.collect()

[(1, int), (2, int), (3, int), (45, int), (6.2, float), ('hello', str)]

In [9]:
def sample_fun(x):
    return x,type(x)

In [10]:
rdd_map = sample_rdd.map(sample_fun)

In [11]:
rdd_map.collect()

[(1, int), (2, int), (3, int), (45, int), (6.2, float), ('hello', str)]

In [12]:
def sample_func(x):
    if type(x) == int or type(x) == float:
        return x,x+1,type(x)
    elif isinstance(x, str):
        return x,x,type(x)
        

In [13]:
rdd_map = sample_rdd.map(sample_func)
rdd_map.collect()

[(1, 2, int),
 (2, 3, int),
 (3, 4, int),
 (45, 46, int),
 (6.2, 7.2, float),
 ('hello', 'hello', str)]

In [14]:
rdd_map = sample_rdd.flatMap(sample_func)
rdd_map.collect()

[1,
 2,
 int,
 2,
 3,
 int,
 3,
 4,
 int,
 45,
 46,
 int,
 6.2,
 7.2,
 float,
 'hello',
 'hello',
 str]

In [15]:
rdd_filter = rdd_map.filter(lambda x: type(x)==int or type(x)==str)

In [16]:
rdd_filter.collect()

[1, 2, 2, 3, 3, 4, 45, 46, 'hello', 'hello']

In [17]:
rdd_filter = rdd_map.filter(lambda x: type(x)==type)
rdd_filter.collect()

[int, int, int, int, float, str]

In [18]:
# Creating 2 partitions for RDD

partitioned_rdd = spark.sparkContext.parallelize([1,2,3,4,5,6,7,8,9,10],3)

In [19]:
partitioned_rdd.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [20]:
partitioned_rdd.getNumPartitions()

3

In [21]:
def fun(itr):
    yield sum(itr)

In [22]:
result = partitioned_rdd.mapPartitions(fun).collect()
result

[6, 15, 34]

In [23]:
def fun(index, iterator):
    return [(index,sum(iterator))]

In [24]:
result = partitioned_rdd.mapPartitionsWithIndex(fun).collect()
result

[(0, 6), (1, 15), (2, 34)]

In [26]:
# Creating a sample of data
# sample(withReplacement, fraction, seed)
sampled = partitioned_rdd.sample(False, 0.4, 40)
print(sampled.collect())

[3, 7, 8, 10]


In [27]:
# Union
rdd1 = spark.sparkContext.parallelize([1,2,3])
rdd2 = spark.sparkContext.parallelize([4,5,6])
result = rdd1.union(rdd2).collect()
result

[1, 2, 3, 4, 5, 6]

In [29]:
# Intersection
rdd1 = spark.sparkContext.parallelize([1,2,3,4])
rdd2 = spark.sparkContext.parallelize([3,4,5,6])
result = rdd1.intersection(rdd2).collect()
result

[3, 4]

In [37]:
# Distinct
rdd = sc.parallelize([1, 2, 1, 3, 2, 4])
result = rdd.distinct().collect()
result

[1, 2, 3, 4]

In [35]:
# distinct([numPartitions])
rdd = sc.parallelize([1, 2, 2, 3, 3, 4, 4, 5], 2)
distinct_rdd = rdd.distinct(numPartitions=4)
print(distinct_rdd.getNumPartitions())
print(distinct_rdd.collect())

4
[4, 1, 5, 2, 3]


In [40]:
# pipe
rdd = sc.parallelize(['hello', 'world', 'how are you'])
result = rdd.pipe('wc -c').collect()
# 'wc -c' is a unix command to measure file size or string length in bytes directly from the command line

In [39]:
result

['       0',
 '       0',
 '       6',
 '       0',
 '       0',
 '       6',
 '       0',
 '      12']